In [0]:
%run ../log/00_audit_logging

In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ---- Configuration ----
CATALOG = "workspace"
SCHEMA = "tmdb"

SILVER_MOVIES = f"{CATALOG}.{SCHEMA}.silver_movies"
DIM_MOVIES   = f"{CATALOG}.{SCHEMA}.dim_movies"
DIM_DATE     = f"{CATALOG}.{SCHEMA}.dim_date"
FACT_PERF    = f"{CATALOG}.{SCHEMA}.fact_movie_performance"

print("Source :", SILVER_MOVIES)
print("Targets :", DIM_MOVIES, "|", DIM_DATE, "|", FACT_PERF)
print("Audit logger loaded :", callable(log_pipeline_execution))

In [0]:
%sql
-- Create Gold Star Schema tables (idempotent). Liquid clustering via CLUSTER BY.

CREATE TABLE IF NOT EXISTS workspace.tmdb.dim_movies (
    movie_id       BIGINT,
    title          STRING,
    original_title STRING,
    overview       STRING,
    release_date   DATE,
    status         STRING,
    tagline        STRING
)
USING DELTA
CLUSTER BY (movie_id);

CREATE TABLE IF NOT EXISTS workspace.tmdb.dim_date (
    date_key     INT,     -- YYYYMMDD e.g. 20080417
    release_date DATE,
    year         INT,
    quarter      INT,
    month        INT,
    day          INT,
    day_of_week  INT      -- 1 = Sunday ... 7 = Saturday (Spark convention)
)
USING DELTA
CLUSTER BY (date_key);

CREATE TABLE IF NOT EXISTS workspace.tmdb.fact_movie_performance (
    movie_id       BIGINT,
    date_key       INT,
    budget         BIGINT,
    revenue        BIGINT,
    profit         BIGINT,          -- revenue - budget
    roi_percentage DOUBLE,          -- (profit / budget) * 100, NULL when budget <= 0
    popularity     DOUBLE,
    vote_average   DOUBLE,
    vote_count    BIGINT,
    batch_id       STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
CLUSTER BY (movie_id, date_key);

In [0]:
silver_df = spark.table(SILVER_MOVIES)

# ---- dim_movies source (movie attributes) ----
dim_movies_src = (
    silver_df
    .select("movie_id", "title", "original_title", "overview",
            "release_date", "status", "tagline")
    .dropDuplicates(["movie_id"])
)

# ---- dim_date source (one row per distinct release_date) ----
# date_key = YYYYMMDD integer derived from release_date
dim_date_src = (
    silver_df
    .filter(F.col("release_date").isNotNull())
    .select("release_date")
    .distinct()
    .select(
        F.date_format("release_date", "yyyyMMdd").cast("int").alias("date_key"),
        F.col("release_date"),
        F.year("release_date").alias("year"),
        F.quarter("release_date").alias("quarter"),
        F.month("release_date").alias("month"),
        F.dayofmonth("release_date").alias("day"),
        F.dayofweek("release_date").alias("day_of_week"),
    )
)

# ---- fact source (one row per movie with a valid release_date) ----
# Movies with NULL release_date cannot join to dim_date and are excluded.
fact_src = (
    silver_df
    .filter(F.col("release_date").isNotNull())
    .select(
        "movie_id",
        F.date_format("release_date", "yyyyMMdd").cast("int").alias("date_key"),
        "budget", "revenue", "popularity", "vote_average", "vote_count",
        "batch_id", "load_timestamp",
    )
    .withColumn("profit", F.col("revenue") - F.col("budget"))
    .withColumn("roi_percentage",
        F.when(F.col("budget") > 0,
               F.round((F.col("revenue") - F.col("budget")) / F.col("budget") * 100, 2))
         .otherwise(F.lit(None).cast("double")))
    .dropDuplicates(["movie_id"])
)

missing_release_dates = silver_df.filter(F.col("release_date").isNull()).count()
print(f"Silver rows read            : {silver_df.count()}")
print(f"dim_movies source rows      : {dim_movies_src.count()}")
print(f"dim_date source rows        : {dim_date_src.count()}")
print(f"fact source rows            : {fact_src.count()}")
print(f"Excluded (null release_date): {missing_release_dates}")

In [0]:
def get_merge_metrics(table_name):
    """Read insert/update counts from the latest Delta history entry (last MERGE)."""
    hist = spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT 1").collect()[0]
    metrics = hist["operationMetrics"] or {}
    return (int(metrics.get("numTargetRowsInserted", 0)),
            int(metrics.get("numTargetRowsUpdated", 0)))

start_time = datetime.now()

try:
    # ---- 1. Idempotent MERGE into dim_movies ----
    (DeltaTable.forName(spark, DIM_MOVIES).alias("target")
        .merge(dim_movies_src.alias("source"),
               "target.movie_id = source.movie_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    dim_ins, dim_upd = get_merge_metrics(DIM_MOVIES)
    print(f"dim_movies MERGED            : inserted={dim_ins}, updated={dim_upd}")

    # ---- 2. Idempotent MERGE into dim_date ----
    (DeltaTable.forName(spark, DIM_DATE).alias("target")
        .merge(dim_date_src.alias("source"),
               "target.date_key = source.date_key")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    date_ins, date_upd = get_merge_metrics(DIM_DATE)
    print(f"dim_date MERGED              : inserted={date_ins}, updated={date_upd}")

    # ---- 3. Idempotent MERGE into fact_movie_performance ----
    (DeltaTable.forName(spark, FACT_PERF).alias("target")
        .merge(fact_src.alias("source"),
               "target.movie_id = source.movie_id AND target.date_key = source.date_key")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    fact_ins, fact_upd = get_merge_metrics(FACT_PERF)
    print(f"fact_movie_performance MERGED: inserted={fact_ins}, updated={fact_upd}")

    rows_inserted = dim_ins + date_ins + fact_ins
    rows_updated  = dim_upd + date_upd + fact_upd

    # ---- 4. OPTIMIZE clustered tables ----
    spark.sql(f"OPTIMIZE {FACT_PERF}")
    spark.sql(f"OPTIMIZE {DIM_MOVIES}")
    print("OPTIMIZE completed on fact_movie_performance and dim_movies")

    end_time = datetime.now()
    duration_s = round((end_time - start_time).total_seconds(), 2)

    log_pipeline_execution(
        layer="Silver-to-Gold",
        load_type="full",
        batch_id="ALL_SILVER_BATCHES",
        parameter=SILVER_MOVIES,
        start_time=start_time,
        end_time=end_time,
        status="SUCCESS",
        rows_inserted=rows_inserted,
        rows_updated=rows_updated,
        rows_quarantined=0,
        error_msg="NONE",
    )
    print(f"Gold load SUCCEEDED in {duration_s}s | inserted={rows_inserted}, updated={rows_updated}")

except Exception as e:
    end_time = datetime.now()
    duration_s = round((end_time - start_time).total_seconds(), 2)

    log_pipeline_execution(
        layer="Silver-to-Gold",
        load_type="full",
        batch_id="ALL_SILVER_BATCHES",
        parameter=SILVER_MOVIES,
        start_time=start_time,
        end_time=end_time,
        status="FAILURE",
        rows_inserted=0,
        rows_updated=0,
        rows_quarantined=0,
        error_msg=str(e),
    )
    print(f"Gold load FAILED after {duration_s}s: {e}")
    raise

In [0]:
%sql
-- Row counts across Gold tables
SELECT 'dim_movies' AS table_name, COUNT(*) AS row_count FROM workspace.tmdb.dim_movies
UNION ALL
SELECT 'dim_date', COUNT(*) FROM workspace.tmdb.dim_date
UNION ALL
SELECT 'fact_movie_performance', COUNT(*) FROM workspace.tmdb.fact_movie_performance;

In [0]:
%sql
-- Star schema join check: top 10 movies by revenue
SELECT f.movie_id, m.title, d.year, d.quarter, d.month,
       f.budget, f.revenue, f.profit, f.roi_percentage,
       f.popularity, f.vote_average, f.vote_count
FROM workspace.tmdb.fact_movie_performance f
JOIN workspace.tmdb.dim_movies  m ON f.movie_id = m.movie_id
JOIN workspace.tmdb.dim_date    d ON f.date_key = d.date_key
ORDER BY f.revenue DESC
LIMIT 10;

In [0]:
%sql
-- Latest audit log entries for the Gold layer
SELECT layer_processed, load_type, batch_id, status,
       rows_inserted, rows_updated, rows_quarantined,
       execution_start_time, execution_end_time, error_message
FROM workspace.tmdb.pipeline_execution_logs
WHERE layer_processed = 'Silver-to-Gold'
ORDER BY execution_end_time DESC
LIMIT 5;